In [11]:
import h5py
import numpy as np
import mudata as md
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [12]:
mdata = md.read_h5mu("/ubc/cs/research/beaver/projects/carlos/spatial_totalvi/data/tonsil/tonsil_pp_svg.h5mu")
mdata

/ubc/cs/research/beaver/projects/carlos/conda_envs/totalvi/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/ubc/cs/research/beaver/projects/carlos/conda_envs/totalvi/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 2492 × 1267
  2 modalities
    RNA:	2492 x 984
      obs:	'leiden'
      uns:	'leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'
      obsm:	'X_pca', 'X_umap', 'pos'
      varm:	'PCs'
      obsp:	'connectivities', 'distances'
    Protein:	2492 x 283
      obs:	'leiden'
      uns:	'leiden', 'leiden_colors', 'neighbors', 'pca', 'umap'
      obsm:	'X_pca', 'X_umap', 'pos'
      varm:	'PCs'
      obsp:	'connectivities', 'distances'

In [13]:
# graph = torch.load("/ubc/cs/research/beaver/projects/carlos/spatial_totalvi/data/graph/graph_k5_conn.pt")
# graph["edge_index"].T.shape

In [14]:
# graph["edge_weight"].shape

In [15]:
rna = mdata.mod['RNA']
prot = mdata.mod['Protein']

In [16]:
rna.obsm["pos"]

array([[29., 42.],
       [44., 36.],
       [34.,  6.],
       ...,
       [23., 48.],
       [ 8., 44.],
       [ 5., 44.]], shape=(2492, 2))

In [17]:
prot.obsm["pos"] 

array([[29., 42.],
       [44., 36.],
       [34.,  6.],
       ...,
       [23., 48.],
       [ 8., 44.],
       [ 5., 44.]], shape=(2492, 2))

In [18]:
from graph import graph_construction

# 1. connectivy
print("Constructing Graph using Connectivity...")
k=100
position = prot.obsm["pos"]
mode = 'connectivity'
e_index_conn, e_weight_conn = graph_construction(
    position, 
    k=k, 
    mode=mode
)
print(f"Edge Index Shape: {e_index_conn.shape}")
print(f"Edge Weight Mean: {e_weight_conn.mean():.4f} ")
graph_data = {
    'edge_index': e_index_conn,
    'edge_weight': e_weight_conn
}
save_path = './graph_k100_conn.pt'
torch.save(graph_data, save_path)

Constructing Graph using Connectivity...
Edge Index Shape: torch.Size([2, 249200])
Edge Weight Mean: 1.0000 


In [19]:
# 2. Similarity (Inverse)
k=100
print("Constructing Graph using simiarity(inverse)...")
position = prot.obsm["pos"]
mode = 'similarity'
e_index_sim, e_weight_sim = graph_construction(
    position, 
    k=k, 
    mode=mode,
    similarity_method='inverse'
)
print(f"Edge Index Shape: {e_index_sim.shape}")
print(f"Edge Weight Mean: {e_weight_sim.mean():.4f} ")
save_path = './graph_k100_sim_inv.pt'
torch.save(graph_data, save_path)

Constructing Graph using simiarity(inverse)...
Edge Index Shape: torch.Size([2, 249200])
Edge Weight Mean: 0.3068 


In [20]:
# 2. Similarity (decay)
k=100
print("Constructing Graph using simiarity(decay)...")
position = prot.obsm["pos"]
mode = 'similarity'
e_index_sim_d, e_weight_sim_d = graph_construction( #sigma might be a parameter here.
    position, 
    k=k, 
    mode=mode,
    similarity_method='gaussian'
)
print(f"Edge Index Shape: {e_index_sim_d.shape}")
print(f"Edge Weight Mean: {e_weight_sim_d.mean():.4f} ")
save_path = './graph_k100_sim_gau.pt'
torch.save(graph_data, save_path)

Constructing Graph using simiarity(decay)...


Similarity Method: Gaussian. Using sigma=4.04
Edge Index Shape: torch.Size([2, 249200])
Edge Weight Mean: 0.6036 
